<a href="https://colab.research.google.com/github/ananthurajeev/GEO5017/blob/main/GEO5017_ConvNeXt_Base.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GEO5017 — Urban Waste Detection: ConvNeXt-Base



## Setup

In [ ]:
!pip install -q timm torch torchvision tqdm scikit-learn matplotlib seaborn Pillow

import os, random, shutil, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from tqdm.notebook import tqdm
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
import timm

from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    f1_score, precision_score, recall_score, accuracy_score,
    roc_auc_score, average_precision_score, precision_recall_curve
)

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Device: {DEVICE}")
if DEVICE == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")


## Mount Drive & Set Paths

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

BASE_DIR    = '/content/drive/MyDrive/Waste/UrbanWaste-images-10k-right' #replace with where the images are located
LABELS_FILE = '/content/drive/MyDrive/Waste/waste.csv' #replace with where the waste.csv file is located
OUTPUT_DIR  = '/content/drive/MyDrive/Waste' #replace with where output is required
CKPT_PATH   = os.path.join(OUTPUT_DIR, 'best_convnext.pt')
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"Base dir : {os.path.isdir(BASE_DIR)}")
print(f"Labels   : {os.path.isfile(LABELS_FILE)}")


## Load Labels & Define Splits

In [ ]:
df = pd.read_csv(LABELS_FILE)
df = df.rename(columns={
    'Parent Folder':   'yearfolder',
    'Sub Folder':      'subfolder',
    'Image File Name': 'filename',
    'Labels':          'label_str',
})

df['label'] = df['label_str'].map({'waste': 1, 'no_waste': 0})

def get_split(yf):
    if yf in ['year_2016', 'year_2017', 'year_2018', 'year_2019']: return 'train'
    if yf in ['year_2020', 'year_2021']:                            return 'val'
    if yf in ['year_2022', 'year_2023']:                            return 'test'
    return 'unknown'

df['split']    = df['yearfolder'].apply(get_split)
df             = df[df['split'] != 'unknown'].reset_index(drop=True)
df['filepath'] = df.apply(
    lambda r: os.path.join(BASE_DIR, r['yearfolder'], r['subfolder'], r['filename']), axis=1
)

print(f"{'Split':<6}  {'Clean':>5}  {'Waste':>5}  {'Total':>5}  {'%Waste':>6}")
print("-" * 36)
for split in ['train', 'val', 'test']:
    s  = df[df['split'] == split]
    nc = (s['label'] == 0).sum()
    nw = (s['label'] == 1).sum()
    print(f"{split:<6}  {nc:>5d}  {nw:>5d}  {len(s):>5d}  {nw/len(s)*100:>5.1f}%")


## Dataset, Augmentation & Balanced Sampler

In [ ]:
IMG_SIZE      = 224
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomVerticalFlip(0.1),
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.3, hue=0.08),
    transforms.RandomRotation(15),
    transforms.RandomGrayscale(0.05),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    transforms.RandomErasing(p=0.2, scale=(0.02, 0.15)),
])

val_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

class WasteDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        try:    img = Image.open(row['filepath']).convert('RGB')
        except: img = Image.new('RGB', (IMG_SIZE, IMG_SIZE))
        return self.transform(img), torch.tensor(row['label'], dtype=torch.long), row['filepath']

train_df = df[df['split'] == 'train'].reset_index(drop=True)
val_df   = df[df['split'] == 'val'].reset_index(drop=True)
test_df  = df[df['split'] == 'test'].reset_index(drop=True)

class_counts   = train_df['label'].value_counts().sort_index().values
sample_weights = np.where(train_df['label'].values == 0,
                          1.0 / class_counts[0],
                          1.0 / class_counts[1])
sampler = WeightedRandomSampler(
    weights=torch.tensor(sample_weights, dtype=torch.float),
    num_samples=len(train_df),
    replacement=True
)

class_weights = torch.tensor(class_counts.sum() / (2.0 * class_counts), dtype=torch.float).to(DEVICE)

BATCH_SIZE   = 32
train_loader = DataLoader(WasteDataset(train_df, train_tf), batch_size=BATCH_SIZE,
                          sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(WasteDataset(val_df,   val_tf),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(WasteDataset(test_df,  val_tf),   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=2, pin_memory=True)

print(f"Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")
print(f"Class weights — clean: {class_weights[0]:.3f}  waste: {class_weights[1]:.3f}")
print(f"Sampler: ~50% waste per batch (from {class_counts[1]/class_counts.sum()*100:.1f}%)")


## ConvNeXt-Base Model

In [ ]:
class ConvNeXtWaste(nn.Module):
    def __init__(self, num_classes=2, dropout=0.3):
        super().__init__()
        self.backbone = timm.create_model(
            'convnext_base.fb_in22k_ft_in1k',
            pretrained=True,
            num_classes=0,
        )
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.LayerNorm(feat_dim),
            nn.Dropout(p=dropout),
            nn.Linear(feat_dim, 256),
            nn.GELU(),
            nn.Dropout(p=dropout),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        return self.head(self.backbone(x))

model = ConvNeXtWaste(num_classes=2, dropout=0.3).to(DEVICE)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters : {trainable:,}")
dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
print(f"Output shape (batch=2): {model(dummy).shape}")


## Training (Focal Loss + Warm-up Cosine LR)

In [ ]:
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        return ((1 - pt) ** self.gamma * ce).mean()

MAX_EPOCHS = 30
PATIENCE   = 7
WARMUP     = 3
LR         = 1e-4
HEAD_LR    = 4e-4

criterion = FocalLoss(alpha=class_weights, gamma=2.0)

optimizer = optim.AdamW([
    {'params': [p for n, p in model.named_parameters() if 'head' not in n], 'lr': LR},
    {'params': model.head.parameters(), 'lr': HEAD_LR},
], weight_decay=1e-4)

def lr_lambda(epoch):
    if epoch < WARMUP:
        return float(epoch + 1) / float(WARMUP)
    progress = float(epoch - WARMUP) / float(max(MAX_EPOCHS - WARMUP, 1))
    return 0.5 * (1.0 + np.cos(np.pi * progress))

scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_acc': [], 'lr': []}
best_val_f1, no_improve = 0.0, 0

print("─" * 65)
print(f" ConvNeXt-Base — max {MAX_EPOCHS} epochs, early-stop patience {PATIENCE}")
print("─" * 65)

for epoch in range(1, MAX_EPOCHS + 1):
    model.train()
    train_loss, nan_batches = 0.0, 0
    for images, labels, _ in tqdm(train_loader, desc=f'Epoch {epoch:02d} train', leave=False):
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        if torch.isnan(loss): nan_batches += 1; continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item()
    train_loss /= max(len(train_loader) - nan_batches, 1)

    model.eval()
    val_loss, all_preds, all_labels_ep = 0.0, [], []
    with torch.no_grad():
        for images, labels, _ in tqdm(val_loader, desc=f'Epoch {epoch:02d} val', leave=False):
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            logits      = model(images)
            val_loss   += criterion(logits, labels).item()
            all_preds.extend(logits.argmax(dim=1).cpu().numpy())
            all_labels_ep.extend(labels.cpu().numpy())

    val_loss /= len(val_loader)
    val_f1   = f1_score(all_labels_ep, all_preds, pos_label=1, zero_division=0)
    val_acc  = accuracy_score(all_labels_ep, all_preds)

    history['lr'].append(optimizer.param_groups[0]['lr'])
    scheduler.step()
    for k, v in zip(['train_loss', 'val_loss', 'val_f1', 'val_acc'],
                    [train_loss, val_loss, val_f1, val_acc]):
        history[k].append(v)

    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        no_improve  = 0
        torch.save(model.state_dict(), CKPT_PATH)
        marker = ' ★ best saved'
    else:
        no_improve += 1
        marker = f' (no improve {no_improve}/{PATIENCE})'

    print(f'Epoch {epoch:02d}  train={train_loss:.4f}  val={val_loss:.4f}  F1={val_f1:.4f}  acc={val_acc:.4f}{marker}')

    if no_improve >= PATIENCE:
        print(f'\nEarly stop — no improvement for {PATIENCE} epochs.')
        break

print(f'\nBest val F1: {best_val_f1:.4f}  |  Checkpoint: {CKPT_PATH}')


## Training Curves

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 4))

axes[0].plot(history['train_loss'], label='Train', color='steelblue')
axes[0].plot(history['val_loss'],   label='Val',   color='coral')
axes[0].axvline(np.argmin(history['val_loss']), linestyle='--', color='gray', alpha=0.5)
axes[0].set_title('Focal Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()

axes[1].plot(history['val_f1'],  label='Val F1 (waste)', color='green')
axes[1].plot(history['val_acc'], label='Val Accuracy',    color='purple')
axes[1].axvline(np.argmax(history['val_f1']), linestyle='--', color='gray', alpha=0.5)
axes[1].set_title('Validation Metrics'); axes[1].set_xlabel('Epoch'); axes[1].legend()

axes[2].plot(history['lr'], color='darkorange')
axes[2].set_title('Learning Rate'); axes[2].set_xlabel('Epoch')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=150)
plt.show()


## Threshold Optimisation on Validation Set

In [ ]:
model.load_state_dict(torch.load(CKPT_PATH, map_location=DEVICE))
model.eval()
print(f"Loaded checkpoint: {CKPT_PATH}")

all_val_labels, all_val_scores, all_val_paths = [], [], []
with torch.no_grad():
    for images, labels, paths in tqdm(val_loader, desc='Val inference'):
        probs = model(images.to(DEVICE)).softmax(dim=1)
        all_val_scores.extend(probs[:, 1].cpu().numpy())
        all_val_labels.extend(labels.numpy())
        all_val_paths.extend(paths)

all_val_labels = np.array(all_val_labels)
all_val_scores = np.array(all_val_scores)

thresholds       = np.arange(0.05, 0.95, 0.01)
f1_scores_thresh = np.array([
    f1_score(all_val_labels, (all_val_scores >= t).astype(int), pos_label=1, zero_division=0)
    for t in thresholds
])

BEST_THRESHOLD  = thresholds[np.argmax(f1_scores_thresh)]
best_f1_thresh  = f1_scores_thresh.max()
print(f"Best threshold : {BEST_THRESHOLD:.2f}")
print(f"Best val F1    : {best_f1_thresh:.4f}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].plot(thresholds, f1_scores_thresh, color='steelblue')
axes[0].axvline(BEST_THRESHOLD, color='red',  linestyle='--', label=f'Best {BEST_THRESHOLD:.2f}')
axes[0].axvline(0.5,            color='gray', linestyle=':',  label='Default 0.50')
axes[0].set_xlabel('Threshold'); axes[0].set_ylabel('F1 (waste)')
axes[0].set_title('Threshold vs F1 on Validation'); axes[0].legend()

pc, rc, _ = precision_recall_curve(all_val_labels, all_val_scores)
ap_val = average_precision_score(all_val_labels, all_val_scores)
axes[1].plot(rc, pc, color='darkorange')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title(f'PR Curve  (AP={ap_val:.3f})')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'threshold_tuning.png'), dpi=150)
plt.show()


## Validation Report

In [ ]:
val_preds_best = (all_val_scores >= BEST_THRESHOLD).astype(int)
print(f"\nValidation Report (threshold={BEST_THRESHOLD:.2f})")
print(classification_report(all_val_labels, val_preds_best, target_names=['clean', 'waste']))

roc_val = roc_auc_score(all_val_labels, all_val_scores)
print(f"ROC-AUC : {roc_val:.4f}")
print(f"Avg Prec: {ap_val:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(
    confusion_matrix(all_val_labels, val_preds_best),
    display_labels=['clean', 'waste']
).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title(f'Validation Confusion Matrix (t={BEST_THRESHOLD:.2f})')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_val.png'), dpi=150)
plt.show()

top100_val_idx = np.argsort(all_val_scores)[::-1][:100]
p100_val       = all_val_labels[top100_val_idx].sum() / 100
print(f"\nP@100 (val): {p100_val:.2f}  ({int(all_val_labels[top100_val_idx].sum())}/100 true waste)")


## Test Set Evaluation

In [ ]:
model.eval()
all_test_labels, all_test_scores, all_test_paths = [], [], []
with torch.no_grad():
    for images, labels, paths in tqdm(test_loader, desc='Test inference'):
        probs = model(images.to(DEVICE)).softmax(dim=1)
        all_test_scores.extend(probs[:, 1].cpu().numpy())
        all_test_labels.extend(labels.numpy())
        all_test_paths.extend(paths)

all_test_labels = np.array(all_test_labels)
all_test_scores = np.array(all_test_scores)
all_test_paths  = np.array(all_test_paths)
test_preds      = (all_test_scores >= BEST_THRESHOLD).astype(int)

print(f"\nTest Report (threshold={BEST_THRESHOLD:.2f})")
print(classification_report(all_test_labels, test_preds, target_names=['clean', 'waste']))

test_f1  = f1_score(all_test_labels, test_preds, pos_label=1, zero_division=0)
test_acc = accuracy_score(all_test_labels, test_preds)
test_pre = precision_score(all_test_labels, test_preds, pos_label=1, zero_division=0)
test_rec = recall_score(all_test_labels, test_preds, pos_label=1, zero_division=0)
roc_test = roc_auc_score(all_test_labels, all_test_scores)
ap_test  = average_precision_score(all_test_labels, all_test_scores)

top100_test_idx = np.argsort(all_test_scores)[::-1][:100]
p100_test       = all_test_labels[top100_test_idx].sum() / 100

print(f"Accuracy  : {test_acc:.4f}")
print(f"Precision : {test_pre:.4f}")
print(f"Recall    : {test_rec:.4f}")
print(f"F1 waste  : {test_f1:.4f}")
print(f"ROC-AUC   : {roc_test:.4f}")
print(f"Avg Prec  : {ap_test:.4f}")
print(f"P@100     : {p100_test:.2f}  ({int(all_test_labels[top100_test_idx].sum())}/100 true waste)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
ConfusionMatrixDisplay(
    confusion_matrix(all_test_labels, test_preds),
    display_labels=['clean', 'waste']
).plot(ax=axes[0], colorbar=False, cmap='Oranges')
axes[0].set_title(f'Test Confusion Matrix (t={BEST_THRESHOLD:.2f})')

axes[1].hist(all_test_scores[all_test_labels == 0], bins=40, alpha=0.6, color='seagreen', label='Clean (GT)')
axes[1].hist(all_test_scores[all_test_labels == 1], bins=40, alpha=0.6, color='crimson',  label='Waste (GT)')
axes[1].axvline(BEST_THRESHOLD, color='black', linestyle='--', label=f'Threshold={BEST_THRESHOLD:.2f}')
axes[1].set_xlabel('P(waste) score'); axes[1].set_ylabel('Count')
axes[1].set_title('Score Distribution by GT Label'); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'test_evaluation.png'), dpi=150)
plt.show()


## Top-100 Breakdown

In [ ]:
top100_labels = all_test_labels[top100_test_idx]
top100_scores = all_test_scores[top100_test_idx]
top100_paths  = all_test_paths[top100_test_idx]

tp = (top100_labels == 1).sum()
fp = (top100_labels == 0).sum()
print(f"Top-100:  TP={tp}  FP={fp}  P@100={tp/100:.2f}")

colors = ['crimson' if l == 1 else 'steelblue' for l in top100_labels]
fig, ax = plt.subplots(figsize=(14, 3))
ax.bar(range(1, 101), top100_scores, color=colors, edgecolor='none')
ax.set_xlabel('Rank'); ax.set_ylabel('P(waste) score')
ax.set_title(f'Top-100 Scores by Rank  (red=TP, blue=FP)  P@100={p100_test:.2f}')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'top100_breakdown.png'), dpi=150)
plt.show()


## Generate Top-100 Submission

In [ ]:
SUBMIT_DIR = os.path.join(OUTPUT_DIR, 'top100_ConvNeXt_Base_submission')
os.makedirs(SUBMIT_DIR, exist_ok=True)

missing = 0
for rank, (path, score, gt) in enumerate(zip(top100_paths, top100_scores, top100_labels), start=1):
    fname = f'rank{rank:03d}_score{score:.3f}_GT{gt}_{Path(path).name}'
    try:
        shutil.copy2(path, os.path.join(SUBMIT_DIR, fname))
    except:
        missing += 1

submission_df = pd.DataFrame({
    'rank':         range(1, 101),
    'filepath':     top100_paths,
    'waste_score':  top100_scores,
    'ground_truth': top100_labels,
    'correct':      (top100_labels == 1).astype(int),
})
csv_path = os.path.join(SUBMIT_DIR, 'top100_ranked.csv')
submission_df.to_csv(csv_path, index=False)

print(f"Copied {100 - missing}/100 images → {SUBMIT_DIR}")
print(f"CSV saved: {csv_path}")


## Visualise Top-20 Detections

In [ ]:
fig, axes = plt.subplots(4, 5, figsize=(18, 13))
for ax, path, score, gt in zip(axes.flat, top100_paths[:20], top100_scores[:20], top100_labels[:20]):
    try: ax.imshow(Image.open(path).convert('RGB'))
    except: pass
    label = 'WASTE' if gt == 1 else 'CLEAN'
    color = 'green'  if gt == 1 else 'red'
    ax.set_title(f'{label}  {score:.3f}', color=color, fontsize=8, fontweight='bold')
    ax.axis('off')
plt.suptitle('Top-20 Detections  (green=correct waste, red=false positive)', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'top20_detections.png'), dpi=150)
plt.show()


## Error Analysis (FP / FN)

In [ ]:
test_result_df = pd.DataFrame({
    'filepath':    all_test_paths,
    'true_label':  all_test_labels,
    'pred_label':  test_preds,
    'waste_score': all_test_scores,
})

fp_df = (test_result_df[(test_result_df['pred_label'] == 1) & (test_result_df['true_label'] == 0)]
         .nlargest(8, 'waste_score'))
fn_df = (test_result_df[(test_result_df['pred_label'] == 0) & (test_result_df['true_label'] == 1)]
         .nsmallest(8, 'waste_score'))

print(f"FP: {len(fp_df)}  FN: {len(fn_df)}")

def show_errors(error_df, title, border_color, save_name):
    n = min(len(error_df), 8)
    if n == 0:
        print(f"No {title} found."); return
    fig, axes = plt.subplots(2, 4, figsize=(16, 7))
    for ax, (_, row) in zip(axes.flat, error_df.head(n).iterrows()):
        try: ax.imshow(Image.open(row['filepath']).convert('RGB'))
        except: pass
        ax.set_title(f"score={row['waste_score']:.3f}", fontsize=8, color=border_color)
        for spine in ax.spines.values():
            spine.set_edgecolor(border_color); spine.set_linewidth(3)
        ax.axis('off')
    for ax in axes.flat[n:]: ax.axis('off')
    plt.suptitle(title, fontsize=12, color=border_color, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, save_name), dpi=150)
    plt.show()

show_errors(fp_df, 'False Positives — predicted WASTE, actually CLEAN', 'orange', 'errors_fp.png')
show_errors(fn_df, 'False Negatives — predicted CLEAN, actually WASTE',  'red',    'errors_fn.png')


## Final Results Table

In [ ]:
results_df = pd.DataFrame({
    'Model':      ['ConvNeXt-Base (val)', 'ConvNeXt-Base (test)'],
    'Accuracy':   [accuracy_score(all_val_labels, val_preds_best), test_acc],
    'F1 (waste)': [f1_score(all_val_labels, val_preds_best, pos_label=1, zero_division=0), test_f1],
    'Precision':  [precision_score(all_val_labels, val_preds_best, pos_label=1, zero_division=0), test_pre],
    'Recall':     [recall_score(all_val_labels, val_preds_best, pos_label=1, zero_division=0), test_rec],
    'P@100':      [p100_val, p100_test],
}).set_index('Model')

print(results_df.round(4).to_string())
results_df.round(4).to_csv(os.path.join(OUTPUT_DIR, 'results.csv'))

fig, ax = plt.subplots(figsize=(10, 3))
sns.heatmap(results_df.astype(float), annot=True, fmt='.3f', cmap='YlGn',
            linewidths=0.5, ax=ax, vmin=0, vmax=1)
ax.set_title('ConvNeXt-Base Results')
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'results_heatmap.png'), dpi=150)
plt.show()

print(f"\nAll outputs saved to: {OUTPUT_DIR}")
